# S6.5 — Query Rewrite Node

Verifies the query rewrite node for the agentic RAG workflow.

**Spec**: `specs/spec-S6.5-rewrite-node/spec.md`  
**Target**: `src/services/agents/nodes/rewrite_query_node.py`

In [1]:
import sys
sys.path.insert(0, "../..")

from src.services.agents.nodes.rewrite_query_node import (
    REWRITE_PROMPT,
    QueryRewriteOutput,
    ainvoke_rewrite_query_step,
)
from src.services.agents.state import AgentState, create_initial_state
from src.services.agents.context import AgentContext

print("Imports OK")

Imports OK


## FR-1: QueryRewriteOutput Model

In [2]:
# Validate structured output model
output = QueryRewriteOutput(rewritten_query="BERT bidirectional transformers NLP", reasoning="Expanded acronym")
assert output.rewritten_query == "BERT bidirectional transformers NLP"
assert output.reasoning == "Expanded acronym"

# Default reasoning
output2 = QueryRewriteOutput(rewritten_query="test")
assert output2.reasoning == ""

# Missing required field
try:
    QueryRewriteOutput(reasoning="no query")
    assert False, "Should have raised"
except Exception:
    pass

print("\n\u2713 QueryRewriteOutput model validates correctly")


✓ QueryRewriteOutput model validates correctly


## FR-2: Rewrite Prompt Template

In [3]:
# Validate prompt template
assert isinstance(REWRITE_PROMPT, str)
assert "{query}" in REWRITE_PROMPT

formatted = REWRITE_PROMPT.format(query="How does BERT work?")
assert "How does BERT work?" in formatted
assert "{query}" not in formatted

# Should mention academic/research context
lower = REWRITE_PROMPT.lower()
assert "academic" in lower or "research" in lower or "paper" in lower

print("Formatted prompt preview:")
print(formatted[:300])
print("\n\u2713 REWRITE_PROMPT template is valid")

Formatted prompt preview:
You are a query optimization specialist for academic paper retrieval.

Your task is to rewrite the following research query to improve retrieval results.
The original query did not return enough relevant academic papers.

Rewriting strategies:
- Expand abbreviations (e.g., "NLP" → "natural language 

✓ REWRITE_PROMPT template is valid


## FR-1: Query Rewrite via LLM (mocked)

In [4]:
from unittest.mock import AsyncMock, MagicMock
from langchain_core.messages import HumanMessage

# Set up mocked LLM provider
mock_provider = MagicMock()
mock_model = MagicMock()
mock_structured = AsyncMock()
mock_structured.ainvoke = AsyncMock(
    return_value=QueryRewriteOutput(
        rewritten_query="How does BERT pre-training and bidirectional transformer encoder work in natural language processing NLP?",
        reasoning="Expanded BERT acronym, added full technical terms, included NLP domain context",
    )
)
mock_model.with_structured_output.return_value = mock_structured
mock_provider.get_langchain_model.return_value = mock_model

context = AgentContext(llm_provider=mock_provider, model_name="test-model")
state = create_initial_state("How does BERT work?")

result = await ainvoke_rewrite_query_step(state, context)

# Verify partial state
assert "rewritten_query" in result
assert "messages" in result
assert "metadata" in result
assert "bidirectional" in result["rewritten_query"]

# Verify HumanMessage appended
assert len(result["messages"]) == 1
assert isinstance(result["messages"][0], HumanMessage)
assert result["messages"][0].content == result["rewritten_query"]

print(f"Original:  {state['original_query']}")
print(f"Rewritten: {result['rewritten_query']}")
print(f"Reasoning: {result['metadata']['rewrite']['reasoning']}")
print("\n\u2713 Query rewrite returns correct partial state with HumanMessage")

Original:  How does BERT work?
Rewritten: How does BERT pre-training and bidirectional transformer encoder work in natural language processing NLP?
Reasoning: Expanded BERT acronym, added full technical terms, included NLP domain context

✓ Query rewrite returns correct partial state with HumanMessage


## FR-1: Uses original_query (prevents semantic drift)

In [5]:
# Simulate a state where a previous rewrite added a different HumanMessage
state2 = create_initial_state("How does BERT work?")
state2["messages"].append(HumanMessage(content="BERT bidirectional transformers NLP"))

mock_structured.ainvoke.reset_mock()
mock_structured.ainvoke = AsyncMock(
    return_value=QueryRewriteOutput(rewritten_query="improved query", reasoning="test")
)

await ainvoke_rewrite_query_step(state2, context)

# Check prompt was called with original_query, not the latest message
call_args = mock_structured.ainvoke.call_args[0][0]
assert "How does BERT work?" in call_args
assert "BERT bidirectional transformers NLP" not in call_args

print(f"Original query in state: {state2['original_query']}")
print(f"Latest message: {state2['messages'][-1].content}")
print(f"Prompt used original_query: {'How does BERT work?' in call_args}")
print("\n\u2713 Rewrite reads original_query to prevent semantic drift")

Original query in state: How does BERT work?
Latest message: BERT bidirectional transformers NLP
Prompt used original_query: True

✓ Rewrite reads original_query to prevent semantic drift


## FR-1: LLM Failure Fallback (keyword expansion)

In [6]:
# Simulate LLM failure
fail_provider = MagicMock()
fail_model = MagicMock()
fail_structured = AsyncMock()
fail_structured.ainvoke = AsyncMock(side_effect=RuntimeError("LLM timeout"))
fail_model.with_structured_output.return_value = fail_structured
fail_provider.get_langchain_model.return_value = fail_model

fail_context = AgentContext(llm_provider=fail_provider, model_name="test-model")
fail_state = create_initial_state("How does BERT work?")

result = await ainvoke_rewrite_query_step(fail_state, fail_context)

# Should fallback to keyword expansion
assert "How does BERT work?" in result["rewritten_query"]
assert "research paper" in result["rewritten_query"].lower() or "arxiv" in result["rewritten_query"].lower()
assert len(result["messages"]) == 1
assert isinstance(result["messages"][0], HumanMessage)

print(f"Fallback query: {result['rewritten_query']}")
print(f"Metadata reasoning: {result['metadata']['rewrite']['reasoning'][:100]}")
print("\n\u2713 LLM failure gracefully falls back to keyword expansion")

Rewrite LLM call failed: LLM timeout — falling back to keyword expansion


Fallback query: How does BERT work? research paper arxiv
Metadata reasoning: LLM rewrite failed, using keyword expansion fallback: LLM timeout

✓ LLM failure gracefully falls back to keyword expansion


## FR-3: Metadata Enrichment

In [7]:
# Verify metadata with attempt tracking
mock_structured.ainvoke = AsyncMock(
    return_value=QueryRewriteOutput(rewritten_query="expanded query", reasoning="added synonyms")
)

state3 = create_initial_state("attention mechanism")
state3["retrieval_attempts"] = 2

result = await ainvoke_rewrite_query_step(state3, context)

meta = result["metadata"]["rewrite"]
assert meta["original_query"] == "attention mechanism"
assert meta["rewritten_query"] == "expanded query"
assert meta["reasoning"] == "added synonyms"
assert meta["attempt_number"] == 2

print(f"Metadata: {meta}")
print("\n\u2713 Metadata enriched with all rewrite details")

Metadata: {'original_query': 'attention mechanism', 'rewritten_query': 'expanded query', 'reasoning': 'added synonyms', 'attempt_number': 2}

✓ Metadata enriched with all rewrite details


## Exported from __init__.py

In [8]:
from pydantic import BaseModel
from src.services.agents.nodes import (
    REWRITE_PROMPT,
    QueryRewriteOutput,
    ainvoke_rewrite_query_step,
)

assert callable(ainvoke_rewrite_query_step)
assert isinstance(REWRITE_PROMPT, str)
assert issubclass(QueryRewriteOutput, BaseModel)

print("\n\u2713 All exports accessible from src.services.agents.nodes")
print("\n=== S6.5 Query Rewrite Node: ALL CHECKS PASSED ===")


✓ All exports accessible from src.services.agents.nodes

=== S6.5 Query Rewrite Node: ALL CHECKS PASSED ===
